# 10 · 同一固定划分：新提取 55 项／提供的 107 项 → 原生 LR → 无标签重载预测

沿用目录提供的 train／val，不为了分数重新挑 seed 或划分。两个输入视图采用同一 SplitPlan、单一固定 LR、相同训练内标准化与阈值 0.5；没有基于 val 挑模型或特征。

0/1 是提供的样例标签，含义未附说明，不命名成恶性／良性。ID 是源记录分组键，不证明独立患者；文件名的“multicenter”也不证明真正外部中心。AUC 只是样例流程的区分能力，不是临床有效性证明。

提供的 107 列提取协议未知，不能把它与本包的 55 列视图当作同参数后端一致性对照。临床表的时间顺序未声明，尤其治疗信息可能是预测时不可用变量，因此本页不自动将它们加入预测器。

## 1. 环境和输入身份

In [ ]:
from pathlib import Path
import hashlib, json, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import ov_radiomics as rad
from ov_radiomics._backend import resolve_backend

if 'TUTORIAL_DATA_ROOT' not in globals() or 'TUTORIAL_RUN_ROOT' not in globals():
    raise RuntimeError('请用本教程的 run_local_sample_tutorials.py --data-root ... --output ... 执行；或先设置这两个路径变量。')
DATA = Path(TUTORIAL_DATA_ROOT).resolve() / 'CT'
output = Path(TUTORIAL_RUN_ROOT).resolve()
output.mkdir(parents=True, exist_ok=False)
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
def save_json(name, value):
    (output / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')
native = resolve_backend('rust')
runtime = dict(backend=rad.backend_info(), interpreter=sys.executable, python=sys.version,
    package_version=rad.__version__, package_path=rad.__file__, native_sha256=sha256(native.__file__),
    implementation_sha256=native.IMPLEMENTATION_SHA256, data_kind='local-image-mask-tabular-no-clinical-validity',
    distribution='local-only: source/redistribution license not established')
assert runtime['backend']['metrics_backend'] == 'rust'
save_json('tutorial-runtime.json', runtime)
display(pd.Series(runtime['backend'], name='actual backend'))

## 2. 精确连接真实记录，保留既有划分

先运行第09篇，或通过 `TUTORIAL_IMAGE_FEATURES` 指定其 features.csv。本页只从特征列选取预测器，不把 ID、label、group 放进模型。影像标签与既有表格标签先逐 ID 比较。

In [ ]:
if 'TUTORIAL_IMAGE_FEATURES' not in globals():
    raise RuntimeError('请先执行09篇，并提供它生成的 features.csv 路径。')
feature_path = Path(TUTORIAL_IMAGE_FEATURES).resolve()
labels = pd.read_csv(DATA/'label_multicenter.csv', dtype=str)
separate = pd.read_csv(DATA/'label.csv', dtype=str)
pd.testing.assert_series_equal(labels.set_index('ID').label.sort_index(), separate.set_index('ID').label.sort_index())
raw_views = {'native55': pd.read_csv(feature_path, dtype={'ID':str}),
             'provided107': pd.read_csv(DATA/'rad_features.csv', dtype={'ID':str})}
source_hashes = {str(path):sha256(path) for path in [feature_path, DATA/'rad_features.csv', DATA/'label.csv', DATA/'label_multicenter.csv']}
save_json('source-hashes.json', source_hashes)
views = {}
for name, frame in raw_views.items():
    assert frame.ID.is_unique and labels.ID.is_unique and set(frame.ID)==set(labels.ID)
    predictors = [x for x in frame.columns if x != 'ID']
    assert len(predictors)==(55 if name=='native55' else 107)
    assert not set(predictors) & {'ID','label','group','patient_id'}
    frame = frame.merge(labels, on='ID', how='inner', validate='one_to_one')
    frame['patient_id'] = frame.ID  # explicit source-record grouping; not confirmed patient linkage
    frame.to_csv(output/(name+'-study.csv'), index=False)
    views[name] = (frame, predictors)
display(labels.groupby(['group','label']).size().unstack(fill_value=0))
save_json('split-assignments.json', labels[['ID','group','label']].to_dict('records'))
protocol = dict(classes=['0','1'], positive_label='1', outer_cv=3, inner_cv=2,
    seed=20261116, model='LR', model_parameters={'lambda':.1,'tolerance':1e-10,'max_iterations':5000},
    scaling='standard', selector=None, threshold=.5, fit_unit='sample', scoring_unit='sample',
    holdout='provided val; not verified external-center validation',
    group_identity='source-record IDs only; hidden patient linkage is unknown')
save_json('protocol.json', protocol)

## 3. 训练内拟合；两个视图共享完整划分

标准化和缺失值处理由 Rust 在训练折拟合；val 不参与这些统计。每个视图只有一个候选 LR，不进行调参搜索。外层三折提供训练 OOF，val 来自最终训练模型。共享 SplitPlan 校验的是全部记录分组，不是碰巧相同的 fold 整数。

In [ ]:
base_data = views['native55'][0]
plan = rad.SplitPlan.from_data(base_data, 'label', task='binary', classes=protocol['classes'],
    id_col='ID', patient_id='patient_id', split='group', outer_cv=3, inner_cv=2, seed=protocol['seed'])
save_json('split-plan.json', plan.to_dict())
results, cohorts, oofs = {}, [], []
for name, (data, predictors) in views.items():
    fit = rad.train_models(data, 'label', predictors, backend='rust', task='binary',
        classes=protocol['classes'], positive_label='1', models=['LR'],
        model_params={'LR':protocol['model_parameters']}, scale='standard', selector=None,
        categorical=[], split_plan=plan, patient_id='patient_id', id_col='ID', split='group',
        cv=3, inner_cv=2, random_state=protocol['seed'], fit_unit='sample', scoring_unit='sample',
        threshold=.5, n_jobs=1)
    fit.split_plan.assert_shared(plan)
    fit.save(output/(name+'-training'))
    fit.models['LR'].save(output/('model-'+name+'.rad'))
    results[name] = fit
    report = rad.evaluate(fit, backend='rust', prediction_unit='sample', ci_method=None)
    report['view'] = name
    cohorts.append(report)
    run = fit.to_dict()['models'][0]
    oof = rad.evaluate(rad.PredictionTable(run['oof']), data=data, target='label',
        backend='rust', prediction_unit='sample', ci_method=None, threshold=.5)
    oof['view'] = name
    oofs.append(oof)
    print(name, 'predictors:',len(predictors), 'SplitPlan:',fit.split_plan.fingerprint)
metrics = pd.concat(cohorts, ignore_index=True)
oof_metrics = pd.concat(oofs, ignore_index=True)
metrics.to_csv(output/'metrics.csv', index=False)
oof_metrics.to_csv(output/'oof-metrics.csv', index=False)
display(metrics)
display(oof_metrics)

## 4. 看固定 val 与 OOF，不拿训练表观分数替代留出

两条曲线只是不同输入视图在这个固定样例划分下的结果。107项来自既有外部表，55项来自本次原生流程；特征定义和前处理不同，不据此宣布某个提取后端更准确。阈值始终0.5，不用val重调。

In [ ]:
val_scores = {}
val_frames = {}
for name, fit in results.items():
    frame = fit.predictions
    held = frame.loc[frame.group=='val'].merge(labels[['ID','label']], on='ID', validate='one_to_one', suffixes=('','_source'))
    if 'label_source' in held:
        assert held.label.astype(str).eq(held.label_source.astype(str)).all()
    held = held.sort_values('ID').reset_index(drop=True)
    val_frames[name] = held
    val_scores[name] = held.probability.to_numpy()
truth = (val_frames['native55'].label.astype(str)=='1').astype(int)
assert val_frames['native55'].ID.tolist()==val_frames['provided107'].ID.tolist()
ax = rad.plot_roc(truth, val_scores, backend='rust', title='Provided val records; no clinical-validity claim')
ax.figure.savefig(output/'validation-roc.png', dpi=150, bbox_inches='tight')
plt.show()
held = val_frames['native55']
ax = rad.plot_calibration(truth, held.probability, n_bins=5, backend='rust')
ax.set_title('Native55 fixed val calibration; small example cohort')
ax.figure.savefig(output/'validation-calibration.png', dpi=150, bbox_inches='tight')
plt.show()
ax = rad.plot_confusion_matrix(truth, held.probability, threshold=.5,
    class_names=['Provided 0','Provided 1'], backend='rust')
ax.figure.savefig(output/'validation-confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 删除标签，重载同一 .rad 再预测

这里只重用同一 val 展示无标签推理，不改称新外部队列。模型中保存的特征顺序和训练内预处理会被复用，不会重新拟合 val。要求保存前后完整 PredictionTable 逐结构完全相等。

In [ ]:
reload_checks = {}
all_predictions = []
for name, fit in results.items():
    data = views[name][0]
    unlabeled = data.loc[data.group=='val'].drop(columns='label').copy()
    restored = rad.load_model(output/('model-'+name+'.rad'))
    before = fit.predict(unlabeled, model='LR')
    after = restored.predict(unlabeled, id_col='ID', patient_id='patient_id', split='group')
    assert before.to_dict()==after.to_dict()
    reload_checks[name] = True
    f=after.dataframe.copy();f['view']=name;all_predictions.append(f)
pd.concat(all_predictions, ignore_index=True).to_csv(output/'unlabeled-predictions.csv', index=False)
assert all(sha256(path)==value for path,value in source_hashes.items())
receipt = dict(status='passed', source_records=len(labels), training_records=int(labels.group.eq('train').sum()),
    validation_records=int(labels.group.eq('val').sum()), feature_views={'native55':55,'provided107':107},
    protocol=protocol, split_plan_sha256=plan.fingerprint, reloaded_predictions_exact=reload_checks,
    input_hashes_unchanged=True, clinical_validity_claim=False, registry_or_public_distribution=False,
    runtime=runtime, outputs={name:sha256(output/name) for name in ['model-native55.rad','model-provided107.rad',
       'metrics.csv','oof-metrics.csv','validation-roc.png','validation-calibration.png','validation-confusion.png','unlabeled-predictions.csv']})
save_json('receipt.json', receipt)
print('Exact model reload checks:', reload_checks)
print('All source inputs unchanged; this completed a local software tutorial, not clinical validation.')